[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/26_lora_solution.ipynb)

# 🟠 Solution: LoRA (Low-Rank Adaptation)（参考版）

## 📋 题目描述

**难度：** Medium

实现线性层的 LoRA（低秩适配）。

LoRA 冻结基础权重并添加可训练的低秩矩阵 A 和 B，以极少的参数实现高效微调。

**签名:** `LoRALinear(in_features, out_features, rank, alpha=1.0)`（nn.Module）

**前向传播:** `forward(x) -> Tensor`
- `x` — 输入张量 (*, in_features)

**返回:** `linear(x) + (x @ A^T @ B^T) * (alpha/rank)`

**约束:**
- 基础 `nn.Linear` 权重必须冻结（requires_grad=False）
- `lora_A`：(rank, in_features)，`lora_B`：(out_features, rank) 初始化为零
- 只有 LoRA 参数应接收梯度

**提示：** 1. `linear.weight/bias` 设 `requires_grad_(False)` 冻结
2. `lora_A`: `(rank, in)`，`lora_B`: `(out, rank)` 初始化为零
3. 前向：`linear(x) + (x @ lora_A.T @ lora_B.T) * (alpha/rank)`


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ SOLUTION

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0):
        super().__init__()
        # ---- Step 1: 创建基础线性层并冻结 ----
        # 基础层的权重在微调时不更新
        # requires_grad_(False) 告诉优化器不要计算和更新这些参数的梯度
        self.linear = nn.Linear(in_features, out_features)
        self.linear.weight.requires_grad_(False)
        self.linear.bias.requires_grad_(False)

        # ---- Step 2: 创建 LoRA 低秩矩阵 ----
        # lora_A: (rank, in_features) — 降维矩阵
        # lora_B: (out_features, rank) — 升维矩阵
        # 参数量: rank × in + out × rank = rank × (in + out)
        # 对比原始参数: in × out，当 rank << min(in,out) 时节省大量参数
        # 例如 in=1024, out=1024, rank=8: 原始 1M 参数，LoRA 仅 16K 参数
        self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))

        # ---- Step 3: 缩放系数 ----
        # alpha/rank 控制 LoRA 更新的幅度
        # alpha 是超参数，通常设为 rank 的倍数
        # 初始化 lora_B 为零 → 初始时 LoRA 输出为零 → 模型行为与原始一致
        self.scaling = alpha / rank

    def forward(self, x):
        # ---- Step 4: 前向传播 ----
        # 基础层输出：x @ W^T + b
        # LoRA 增量：x @ A^T @ B^T × scaling
        # 数学上：(x @ A^T) 得到 (batch, rank) 的低维表示
        #         (x @ A^T @ B^T) 映射回 (batch, out_features)
        # 最终输出 = 基础输出 + LoRA 增量
        return self.linear(x) + (x @ self.lora_A.T @ self.lora_B.T) * self.scaling


In [ ]:
lora = LoRALinear(in_features=256, out_features=256, rank=8)
x = torch.randn(2, 10, 256)
print('Output:', lora(x).shape)
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
total = sum(p.numel() for p in lora.parameters())
print(f'Trainable: {trainable}/{total} ({100*trainable/total:.1f}%)')


In [ ]:
from torch_judge import check
check('lora')


## 📝 核心思路总结

1. **低秩分解**：ΔW = B × A，参数量从 O(in×out) 降到 O(rank×(in+out))
2. **冻结基础层**：只训练 LoRA 参数，预训练知识不丢失
3. **零初始化**：lora_B 初始化为零，保证训练起始点 = 预训练模型
4. **缩放系数**：alpha/rank 控制 LoRA 更新的强度
